# S0.7 · R-C（粤港澳大湾区个人信息跨境流动标准合同）推演

S0.5 曾判断「R-B 与 R-C 在敏感个人信息口径下不可选」。
**本步查证后推翻了其中关于 R-C 的部分**：大湾区标准合同**免除数量门槛限制**。

依据：香港数字政策办公室公布的便利措施——
「《大湾区标准合同》还免除了个人信息处理者于内地规定下，跨境传输个人信息数量上的限制」；
「简化个人信息保护影响评估（PIA）的重点评估内容，由 6 项减至 3 项」。

若适用，形态 A（双向 L3）就不必走数据出境安全评估。**但适用性有三项前提，目前全部未确认。**

In [1]:
import hashlib
import pathlib
import subprocess
import sys

import pandas as pd
import yaml

HASH_PREFIX_LEN = 12

REPO_ROOT = pathlib.Path.cwd()
while not (REPO_ROOT / "AGENTS.md").exists() and REPO_ROOT != REPO_ROOT.parent:
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT))

CONFIG_PATH = REPO_ROOT / "modules/m0_compliance/configs/s0_7_route_c.yaml"
cfg = yaml.safe_load(CONFIG_PATH.read_text(encoding="utf-8"))
config_hash = hashlib.sha256(CONFIG_PATH.read_bytes()).hexdigest()[:HASH_PREFIX_LEN]
git_sha = subprocess.check_output(["git", "rev-parse", "--short", "HEAD"], text=True).strip()

print("config:", CONFIG_PATH.relative_to(REPO_ROOT), "sha256:" + config_hash)
print("seed:", cfg["seed"])
print("git:", git_sha)
print("step:", cfg["step_id"], "| 路线:", cfg["route"], cfg["route_name"])

config: modules/m0_compliance/configs/s0_7_route_c.yaml sha256:63f1dd21c118
seed: 42
git: 46d99a9
step: S0.7 | 路线: R-C 粤港澳大湾区（内地、香港）个人信息跨境流动标准合同


In [2]:
from modules.m0_compliance.components.export_mechanism import required_mechanism

thresholds = cfg["thresholds"]
records = []
for form in cfg["l3_forms"]:
    for n in cfg["scale_points"]:
        for gba in (False, True):
            records.append({
                "L3 形态": form["id"],
                "含敏感个人信息": "是" if form["cn_to_hk_sensitive"] else "否",
                "涉及人数": n,
                "适用大湾区标准合同": "是" if gba else "否",
                "所需机制": required_mechanism(n, form["cn_to_hk_sensitive"], thresholds, gba_eligible=gba),
            })

matrix = pd.DataFrame(records)
matrix.to_csv(REPO_ROOT / "modules/m0_compliance/results/route_C_mechanism_matrix.csv", index=False)
print(matrix.to_string(index=False))

L3 形态 含敏感个人信息    涉及人数 适用大湾区标准合同           所需机制
  形态A       是    5000         否  标准合同或个人信息保护认证
  形态A       是    5000         是 大湾区标准合同（免数量门槛）
  形态A       是   30000         否       数据出境安全评估
  形态A       是   30000         是 大湾区标准合同（免数量门槛）
  形态A       是  150000         否       数据出境安全评估
  形态A       是  150000         是 大湾区标准合同（免数量门槛）
  形态A       是 1200000         否       数据出境安全评估
  形态A       是 1200000         是 大湾区标准合同（免数量门槛）
  形态B       否    5000         否 豁免（无需申报/订立/认证）
  形态B       否    5000         是 大湾区标准合同（免数量门槛）
  形态B       否   30000         否 豁免（无需申报/订立/认证）
  形态B       否   30000         是 大湾区标准合同（免数量门槛）
  形态B       否  150000         否  标准合同或个人信息保护认证
  形态B       否  150000         是 大湾区标准合同（免数量门槛）
  形态B       否 1200000         否       数据出境安全评估
  形态B       否 1200000         是 大湾区标准合同（免数量门槛）


In [3]:
print("三项适用前提（全部未确认）：")
for p in cfg["gba_prerequisites"]:
    print()
    print("  [%s] %s —— %s" % (p["id"], p["name"], p["status"]))
    print("       要求：%s" % p["requirement"])
    print("       备注：%s" % p["note"])

unconfirmed = [p["id"] for p in cfg["gba_prerequisites"] if p["status"] != "已确认"]
print()
print("未确认前提数：%d / %d" % (len(unconfirmed), len(cfg["gba_prerequisites"])))
print("→ 在三项全部确认之前，**不得把 R-C 当作可依赖的路径**（取最严解释）。")

三项适用前提（全部未确认）：

  [P1] 注册地要求 —— 未确认
       要求：个人信息处理者须注册于大湾区内地九市（穗深珠佛惠莞中江肇）
       备注：汇丰银行（中国）有限公司总部注册地不在九市之内；分行不具独立法人资格，能否满足「注册于」存疑

  [P2] 重要数据排除 —— 未确认
       要求：排除被相关部门、地区告知或公开发布为重要数据的个人信息
       备注：金融行业数据是否被认定为重要数据，须查证行业目录

  [P3] 敏感个人信息是否在免数量门槛范围内 —— 未确认
       要求：官方便利措施表述为「免除跨境传输个人信息数量上的限制」
       备注：字面覆盖敏感个人信息，但指引条文原文未明示；取最严解释前不得依赖

未确认前提数：3 / 3
→ 在三项全部确认之前，**不得把 R-C 当作可依赖的路径**（取最严解释）。


## 结论

**若 R-C 适用，形态 A 的合规成本与周期会大幅下降**：从数据出境安全评估（法定约 57 个工作日、
端到端 6–12 个月、成本档高）降到大湾区标准合同（备案 10 个工作日、查验 10 个工作日、
PIA 重点内容由 6 项减至 3 项、成本档中低）。

**但三项前提目前全部未确认，其中 P1 是最实际的障碍**：
大湾区标准合同要求个人信息处理者**注册于**大湾区内地九市，
而汇丰银行（中国）有限公司的注册地不在九市之内；分行不具独立法人资格，
能否满足「注册于」的要求存疑。**这一条不解决，R-C 无从谈起。**

因此本步的结论不是「R-C 可行」，而是：

> **R-C 是形态 A 唯一可能可行的轻量路径，它的可行性完全取决于一个此前没人问过的问题——
> 内地侧的签约主体到底是谁、注册在哪里。**

这个问题需要需求方回答，技术侧无法自行判定。